## 데이터셋 개요

총 1000개의 샘플로 구성됨

[각 샘플의 기본 구조]

* user_id
* item_id
* 여러 사용자/아이템/도메인 feature
* 정답 label

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [2]:
# df= load_dataset("TAAC2026/data_sample_1000")
df= load_dataset("TAAC2026/data_sample_1000")['train'].to_pandas()
df = df.copy()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

demo_1000.parquet:   0%|          | 0.00/40.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

# 데이터 전처리

* label_type을 정답값으로 사용

* 0/1 binary classification을 적용하기 위해 이진화함.

#### (일단) 제외한 컬럼

* timestamp
* label_time

In [3]:
# 데이터 이진화( 기존 [1,2] -> [0,1]로 변환)
df['label_type'] = (df['label_type'] == 2).astype(np.float32)

In [4]:
df['label_type'].value_counts()

,count
label_type,
0.0,876
1.0,124


In [5]:
# timestamp 일단 드랍
drop_cols = ['timestamp','label_time']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

In [6]:
# label 먼저 저장
y = df['label_type'].values

# label 제거
df = df.drop(columns=['label_type'])

## 데이터 타입별로 분류(일단 미리했어요)

#### sequence feature 전처리

* ndarray면 list로 변환
* list가 아니면 빈 리스트로 처리
* 각 sequence를 최대 길이 50으로 맞춤
* 길이가 50보다 길면 뒤 50개만 사용
* 짧으면 앞쪽을 0으로 padding




In [7]:
seq_cols = [col for col in df.columns
            if isinstance(df[col].iloc[0], (list, np.ndarray))]
print(seq_cols)

['user_int_feats_15', 'user_int_feats_60', 'user_int_feats_62', 'user_int_feats_63', 'user_int_feats_64', 'user_int_feats_65', 'user_int_feats_66', 'user_int_feats_80', 'user_int_feats_89', 'user_int_feats_90', 'user_int_feats_91', 'user_dense_feats_61', 'user_dense_feats_62', 'user_dense_feats_63', 'user_dense_feats_64', 'user_dense_feats_65', 'user_dense_feats_66', 'user_dense_feats_87', 'user_dense_feats_89', 'user_dense_feats_90', 'user_dense_feats_91', 'item_int_feats_11', 'domain_a_seq_38', 'domain_a_seq_39', 'domain_a_seq_40', 'domain_a_seq_41', 'domain_a_seq_42', 'domain_a_seq_43', 'domain_a_seq_44', 'domain_a_seq_45', 'domain_a_seq_46', 'domain_b_seq_67', 'domain_b_seq_68', 'domain_b_seq_69', 'domain_b_seq_70', 'domain_b_seq_71', 'domain_b_seq_72', 'domain_b_seq_73', 'domain_b_seq_74', 'domain_b_seq_75', 'domain_b_seq_76', 'domain_b_seq_77', 'domain_b_seq_78', 'domain_b_seq_79', 'domain_b_seq_88', 'domain_c_seq_27', 'domain_c_seq_28', 'domain_c_seq_29', 'domain_c_seq_30', 'dom

In [8]:
cat_cols = []

for col in df.columns:
    if col in seq_cols:
        continue
    if df[col].dtype in ['int64', 'int32']:
        cat_cols.append(col)
print(cat_cols)

['user_id', 'item_id', 'user_int_feats_1']


In [9]:
num_cols = []

for col in df.columns:
    if col in seq_cols or col in cat_cols:
        continue
    if df[col].dtype in ['float64', 'float32']:
        num_cols.append(col)
print(num_cols)

['user_int_feats_3', 'user_int_feats_4', 'user_int_feats_48', 'user_int_feats_49', 'user_int_feats_50', 'user_int_feats_51', 'user_int_feats_52', 'user_int_feats_53', 'user_int_feats_54', 'user_int_feats_55', 'user_int_feats_56', 'user_int_feats_57', 'user_int_feats_58', 'user_int_feats_59', 'user_int_feats_82', 'user_int_feats_86', 'user_int_feats_92', 'user_int_feats_93', 'user_int_feats_94', 'user_int_feats_95', 'user_int_feats_96', 'user_int_feats_97', 'user_int_feats_98', 'user_int_feats_99', 'user_int_feats_100', 'user_int_feats_101', 'user_int_feats_102', 'user_int_feats_103', 'user_int_feats_104', 'user_int_feats_105', 'user_int_feats_106', 'user_int_feats_107', 'user_int_feats_108', 'user_int_feats_109', 'item_int_feats_5', 'item_int_feats_6', 'item_int_feats_7', 'item_int_feats_8', 'item_int_feats_9', 'item_int_feats_10', 'item_int_feats_12', 'item_int_feats_13', 'item_int_feats_16', 'item_int_feats_81', 'item_int_feats_83', 'item_int_feats_84', 'item_int_feats_85']


In [10]:
# 데이터 상태 확인용
# df['user_int_feats_65'].apply(type).value_counts()

In [11]:
# sequence 처리
def fix_seq(x):
    if isinstance(x, np.ndarray):
        return x.tolist()
    elif isinstance(x, list):
        return x
    else:
        return []

for col in seq_cols:
    df[col] = df[col].apply(fix_seq)

MAX_LEN = 50

def pad_sequence(seq):
    seq = seq[-MAX_LEN:]
    return [0] * (MAX_LEN - len(seq)) + seq

for col in seq_cols:
    df[col] = df[col].apply(pad_sequence)

In [12]:
user_enc = LabelEncoder()
item_enc = LabelEncoder()

user_enc.fit(df['user_id'])
item_enc.fit(df['item_id'])

df['user_id'] = user_enc.transform(df['user_id'])
df['item_id'] = item_enc.transform(df['item_id'])

In [13]:
num_cols = [c for c in num_cols if c != 'label_type']

In [14]:
# cat 처리(int변환)
for col in cat_cols:
    df[col] = df[col].fillna(0).astype(int)

# sequence numpy 변환
X_seq = {col: np.array(df[col].tolist()) for col in seq_cols}

# label
X_cat = df[cat_cols].values

# num
df[num_cols] = df[num_cols].fillna(0)


In [15]:
# 스케일링
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

## 데이터 분리
#### (위에서 타입 분류한 건 아직 적용 안 했음)

* Train: 800개
* Validation: 200개

`stratify=y`를 사용해서 train/validation에서도 label 비율이 원본과 비슷하게 유지되도록 분리

In [16]:
class RecDataset(Dataset):
    def __init__(self, df, num_cols, y):
        self.user = df['user_id'].values
        self.item = df['item_id'].values
        self.num = df[num_cols].values if len(num_cols) > 0 else np.zeros((len(df), 0))
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            'user': torch.tensor(self.user[idx], dtype=torch.long),
            'item': torch.tensor(self.item[idx], dtype=torch.long),
            'num': torch.tensor(self.num[idx], dtype=torch.float32),
            'label': torch.tensor(self.y[idx], dtype=torch.float32)
        }

In [17]:
from sklearn.model_selection import train_test_split
indices = np.arange(len(df))

train_idx, valid_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=y
)

train_df = df.iloc[train_idx].reset_index(drop=True)
valid_df = df.iloc[valid_idx].reset_index(drop=True)

y_train = y[train_idx]
y_valid = y[valid_idx]

print(train_df.shape, valid_df.shape)

(800, 117) (200, 117)


In [18]:
train_dataset = RecDataset(train_df, num_cols, y_train)
valid_dataset = RecDataset(valid_df, num_cols, y_valid)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

## Simple Baseline Model


#### 사용한 핵심 input

* user_id embedding
* item_id embedding
* 수치형 feature(num_cols)


#### baseline 결과

* Epoch 마다 train loss가 줄어들고 있음

Validation 성능
* AUC: 0.6448


In [19]:
class SimpleRecModel(nn.Module):
    def __init__(self, n_users, n_items, embed_dim, num_features):
        super().__init__()

        self.user_emb = nn.Embedding(n_users, embed_dim)
        self.item_emb = nn.Embedding(n_items, embed_dim)

        self.fc = nn.Sequential(
            nn.Linear(embed_dim * 2 + num_features, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, user, item, num):
        user_e = self.user_emb(user)
        item_e = self.item_emb(item)

        if num.dim() == 1:
            num = num.unsqueeze(1)

        x = torch.cat([user_e, item_e, num], dim=1)
        out = self.fc(x)

        return out.squeeze(1)

In [23]:
model = SimpleRecModel(
    n_users=df['user_id'].nunique(),
    n_items=df['item_id'].nunique(),
    embed_dim=32,
    num_features=len(num_cols)
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# pos_weight 적용 (불균형 대응)
pos_weight = torch.tensor([876/124])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
# criterion = nn.BCEWithLogitsLoss()

In [24]:
for epoch in range(10):
    model.train()
    total_loss = 0

    for batch in train_loader:
        logits = model(
            batch['user'],
            batch['item'],
            batch['num']
        )

        loss = criterion(logits, batch['label'])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

Epoch 1, Loss: 1.2083
Epoch 2, Loss: 1.0869
Epoch 3, Loss: 0.9921
Epoch 4, Loss: 0.8904
Epoch 5, Loss: 0.7788
Epoch 6, Loss: 0.6645
Epoch 7, Loss: 0.5573
Epoch 8, Loss: 0.4585
Epoch 9, Loss: 0.3727
Epoch 10, Loss: 0.3034


In [25]:
from sklearn.metrics import roc_auc_score

preds = []
labels = []

model.eval()
with torch.no_grad():
    for batch in valid_loader:
        logits = model(
            batch['user'],
            batch['item'],
            batch['num']
        )
        pred = torch.sigmoid(logits)

        preds.extend(pred.cpu().numpy())
        labels.extend(batch['label'].cpu().numpy())

print("AUC:", roc_auc_score(labels, preds))

AUC: 0.6448
